In [1]:
!pip install spotipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.5/261.5 kB 6.0 MB/s eta 0:00:00


In [2]:
!pip install keras

In [3]:
!pip install tensorflow

In [4]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from sklearn.preprocessing import StandardScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Conv1D, MaxPooling1D, Dense

In [5]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

client_id = '94f4001040da43f388cf6ef2f1c1ab27'
client_secret = '01b12c28fd154a1aadcf6519e127b1f3'

credentials = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=credentials)

In [6]:
import requests
import base64

client_id = '94f4001040da43f388cf6ef2f1c1ab27'
client_secret = '01b12c28fd154a1aadcf6519e127b1f3'

# Encode the client ID and secret
credentials = f"{client_id}:{client_secret}"
encoded_credentials = base64.b64encode(credentials.encode()).decode()

# Set up the authentication request
auth_url = 'https://accounts.spotify.com/api/token'
auth_headers = {
    'Authorization': f'Basic {encoded_credentials}'
}
auth_data = {
    'grant_type': 'client_credentials'
}

# Make the request
response = requests.post(auth_url, headers=auth_headers, data=auth_data)

if response.status_code == 200:
    token = response.json()['access_token']
    print(f"Access Token: {token}")
else:
    print(f"Error: {response.status_code} - {response.text}")


Access Token: BQA-ThjEMIgTAorA647l8nGNCyxpfngc5tgxC8rz5m15lb3ceJpoxtWuIYGGRRXsFNTJO6uO3Tc1oJNB_1OGh1ErtP8vNbrB52cTiBvHDEgkfzrtbAk


In [7]:
# Set up Spotipy with your access token
access_token = 'BQC4o-z-04yygfUBQRGlFrckawPtlftr3YIcXljVk-KYILnOQAgbuAooy897hyHjTLh_u43m1kYOZ9HriqMqCNJNq2hXoGXIn_Rgv1kp9B-p-acqxws'
sp = spotipy.Spotify(auth=access_token)

In [8]:
# Set up Spotipy with your access token
sp = spotipy.Spotify(auth=access_token)

In [9]:
scope = 'user-top-read user-library-read user-read-recently-played'

In [10]:
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id='94f4001040da43f388cf6ef2f1c1ab27',
                                               client_secret='01b12c28fd1543a1aadcf6519e127b1f',
                                               redirect_uri='https://localhost:8888/callback',
                                               scope='user-top-read user-library-read'))

In [11]:
def fetch_user_top_tracks(user_id, limit=20):
    results = sp.current_user_top_tracks(limit=50, offset=0, time_range='medium_term')
    tracks = results['items']
    return tracks

In [12]:
import requests
import base64

# Replace with your own Client ID and Client Secret
CLIENT_ID = '94f4001040da43f388cf6ef2f1c1ab27'
CLIENT_SECRET = '01b12c28fd154a1aadcf6519e127b1f3'

# Base64 encode the client ID and client secret
client_credentials = f"{CLIENT_ID}:{CLIENT_SECRET}"
client_credentials_base64 = base64.b64encode(client_credentials.encode())

# Request the access token
token_url = 'https://accounts.spotify.com/api/token'
headers = {
    'Authorization': f'Basic {client_credentials_base64.decode()}'
}
data = {
    'grant_type': 'client_credentials'
}
response = requests.post(token_url, data=data, headers=headers)

if response.status_code == 200:
    access_token = response.json()['access_token']
    print("Access token obtained successfully.")
else:
    print("Error obtaining access token.")
    exit()

Access token obtained successfully.


In [13]:
def get_trending_playlist_data(playlist_id, access_token):
    # Set up Spotipy with the access token
    sp = spotipy.Spotify(auth=access_token)

    # Get the tracks from the playlist
    playlist_tracks = sp.playlist_tracks(playlist_id, fields='items(track(id, name, artists, album(id, name)))')

    # Extract relevant information and store in a list of dictionaries
    music_data = []
    for track_info in playlist_tracks['items']:
        track = track_info['track']
        track_name = track['name']
        artists = ', '.join([artist['name'] for artist in track['artists']])
        album_name = track['album']['name']
        album_id = track['album']['id']
        track_id = track['id']

        # Get audio features for the track
        audio_features = sp.audio_features(track_id)[0] if track_id != '3r3PiLxBdeFCuxZYx3jGjx' else None

        # Get release date of the album
        try:
            album_info = sp.album(album_id) if album_id != '68e0906f9eb34de3' else None
            release_date = album_info['release_date'] if album_info else None
        except:
            release_date = None

        # Get popularity of the track
        try:
            track_info = sp.track(track_id) if track_id != '3r3PiLxBdeFCuxZYx3jGjx' else None
            popularity = track_info['popularity'] if track_info else None
        except:
            popularity = None

        # Add additional track information to the track data
        track_data = {
            'Track Name': track_name,
            'Artists': artists,
            'Album Name': album_name,
            'Album ID': album_id,
            'Track ID': track_id,
            'Popularity': popularity,
            'Release Date': release_date,
            'Duration (ms)': audio_features['duration_ms'] if audio_features else None,
            'Explicit': track_info.get('explicit', None),
            'External URLs': track_info.get('external_urls', {}).get('spotify', None),
            'Danceability': audio_features['danceability'] if audio_features else None,
            'Energy': audio_features['energy'] if audio_features else None,
            'Key': audio_features['key'] if audio_features else None,
            'Loudness': audio_features['loudness'] if audio_features else None,
            'Mode': audio_features['mode'] if audio_features else None,
            'Speechiness': audio_features['speechiness'] if audio_features else None,
            'Acousticness': audio_features['acousticness'] if audio_features else None,
            'Instrumentalness': audio_features['instrumentalness'] if audio_features else None,
            'Liveness': audio_features['liveness'] if audio_features else None,
            'Valence': audio_features['valence'] if audio_features else None,
            'Tempo': audio_features['tempo'] if audio_features else None,
            # Add more attributes as needed
        }

        music_data.append(track_data)

    # Create a pandas DataFrame from the list of dictionaries
    df = pd.DataFrame(music_data)

    return df

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth

CLIENT_ID = '94f4001040da43f388cf6ef2f1c1ab27'
CLIENT_SECRET = '01b12c28fd154a1aadcf6519e127b1f3'

# Define the scope for the access token
scope = "user-top-read user-library-read user-read-recently-played"  # Include the necessary scope

# Create a SpotifyOAuth object
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,
                                               client_secret=client_secret,
                                               redirect_uri="https://open.spotify.com/playlist/3r3PiLxBdeFCuxZYx3jGjx",
                                               scope=scope))

# Get the access token
access_token = sp.auth_manager.get_access_token(as_dict=False)  # Get the access token directly

# Use the access token in your API calls
def get_trending_playlist_data(playlist_id, access_token):
    # Set up Spotipy with the access token
    sp = spotipy.Spotify(auth=access_token)


playlist_id = '3r3PiLxBdeFCuxZYx3jGjx'

# Call the function to get the music data from the playlist and store it in a DataFrame
music_df = get_trending_playlist_data(playlist_id, access_token)

In [ ]:
music_df

In [ ]:
music_df.isnull().sum()

In [ ]:
# Assuming you have collected data into a DataFrame called music_df
data = music_df[['Track Name', 'Album Name', 'Artists', 'Release Date', 'Duration (ms)', 'Popularity',
                 'Acousticness', 'Danceability', 'Energy', 'Instrumentalness', 'Liveness', 'Loudness',
                 'Speechiness', 'Tempo']]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Normalize the feature columns
scaler = MinMaxScaler()
music_df[['Duration (ms)', 'Popularity', 'Acousticness', 'Danceability', 'Energy', 'Instrumentalness', 'Liveness', 'Loudness', 'Speechiness', 'Tempo']] = scaler.fit_transform(data[['Duration (ms)', 'Popularity', 'Acousticness', 'Danceability', 'Energy', 'Instrumentalness', 'Liveness', 'Loudness', 'Speechiness', 'Tempo']])

In [ ]:
 scaler.fit_transform(data[['Duration (ms)', 'Popularity', 'Acousticness', 'Danceability',
                               'Energy', 'Instrumentalness', 'Liveness', 'Loudness',
                               'Speechiness', 'Tempo']])

In [ ]:
# Function to calculate weighted popularity scores based on release date
def calculate_weighted_popularity(release_date):
    release_date = datetime.strptime(release_date, '%Y-%m-%d')
    time_span = datetime.now() - release_date
    weight = 1 / (time_span.days + 1)
    return weight

In [ ]:
def create_sequences(data, sequence_length):
    sequences = []
    targets = []
    for i in range(len(data) - sequence_length):
        sequences.append(data[i:i+sequence_length])
        targets.append(data[i+sequence_length])
    return np.array(sequences), np.array(targets)

In [ ]:
import numpy as np

sequence_length = 50
data_array = np.array(data.values)
sequences, targets = create_sequences(data_array, sequence_length)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [ ]:
sequence_length = 50

In [ ]:
data_array = data_array.reshape(1, -1)

In [ ]:
music_df.dtypes

In [ ]:
# Prepare data for LSTM
sequence_length = 10
X, y = ["Artists"], ["Track Name"]

# Select only numeric columns for normalization
numeric_data = data.select_dtypes(include=['number'])

# Normalize the numeric data
normalized_data = (numeric_data - numeric_data.mean()) / numeric_data.std()

In [ ]:
# Prepare data for LSTM
sequence_length = 10
X, y = [], []
for i in range(len(normalized_data) - sequence_length):
    X.append(normalized_data.iloc[i:i+sequence_length].values)
    y.append(normalized_data.iloc[i+sequence_length].values)

X, y = np.array(X), np.array(y)

# Reshape for LSTM input
X = X.reshape((X.shape[0], sequence_length, X.shape[2]))

In [ ]:
# Reshape the input data to 3D
X = X.reshape((X.shape[0], sequence_length, 10))

In [ ]:
X = np.random.rand(100, 10, 9)
y = np.random.rand(100,10, 9)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
y_train.shape

In [ ]:
X_train.shape

In [ ]:
X.shape

In [ ]:
y = y.reshape(-1,100)

In [ ]:
y.shape

In [ ]:
X_test.shape

In [ ]:
y = y.reshape(-1,10,100)

In [ ]:
y.shape

In [ ]:
X = X.reshape(-1,10,100)

In [ ]:
X.shape

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X = X.reshape(-1,10,50)

In [ ]:
X.shape

In [ ]:
y = y.reshape(-1,10,50)

In [ ]:
y.shape

In [ ]:
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
# Define the model
from keras.models import Sequential
from keras.layers import LSTM, Dense

model = Sequential()  # Use Sequential for stacking layers
model.add(LSTM(50, return_sequences=True, input_shape=(sequence_length, data_array.shape[1])))
model.add(LSTM(50))
model.add(Dense(10))

In [ ]:
model = Sequential()

# LSTM layer with return_sequences=True to output sequences
model.add(LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(LSTM(50, return_sequences=True))
model.add(Dense(X_train.shape[2]))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Recommendation function with cosine similarity
def recommend_track(model, input_sequence, all_tracks_features):
    # Predict the features for the input sequence
    predicted_features = model.predict(np.array([input_sequence]))

    # Calculate cosine similarity between the predicted features and all tracks in the dataset
    similarities = cosine_similarity(predicted_features, all_tracks_features)

    # Find the index of the most similar track
    most_similar_index = np.argmax(similarities)

    # Return the most similar track's features
    return all_tracks_features[most_similar_index], similarities[0, most_similar_index]

In [ ]:
print(cosine_similarity)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Assuming 'music_df' is your DataFrame with audio features

song1_features = music_df.loc[0, ['Duration (ms)', 'Popularity', 'Acousticness']].values
song2_features = music_df.loc[1, ['Duration (ms)', 'Popularity', 'Acousticness']].values

similarity = cosine_similarity([song1_features], [song2_features])
print(similarity[0][0])

In [ ]:
import numpy as np

input_sequence = np.random.random((10, 100))

# Reshape to match the expected input shape for the model (1, sequence_length, number_of_features)
input_sequence = input_sequence.reshape((1, input_sequence.shape[0], input_sequence.shape[1]))

# Predict with the model
recommended_features = model.predict(input_sequence)
print(recommended_features)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
def evaluate_model(model, X_test, y_test):
  """
  Evaluates the given model using the provided test data.

  Args:
    model: The trained model to evaluate.
    X_test: The test input data.
    y_test: The corresponding test labels.
  """
  loss = model.evaluate(X_test, y_test)
  print("Test Loss:", loss)

# Evaluate model
evaluate_model(model, X_test, y_test)

In [ ]:
import pandas as pd
from sklearn import datasets

# Load music_df datasets
music_df
X = music_df.select_dtypes(include=['number'])
y = music_df['Track Name']

In [ ]:
# Select only two classes (0 and 1) for binary classification
binary_mask = music_df['Track Name'].isin(music_df['Track Name'].unique()[:2])
X_binary = music_df[binary_mask].drop('Track Name', axis=1)
y_binary = music_df[binary_mask]['Track Name']

In [ ]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create the logistic regression model

log_reg=LogisticRegression()

In [ ]:
X = music_df[['Duration (ms)', 'Popularity', 'Acousticness', 'Danceability', 'Energy', 'Instrumentalness', 'Liveness', 'Loudness', 'Speechiness', 'Tempo']]

In [ ]:
y = music_df['Track Name']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train the model
log_reg.fit(X_train,y_train)

In [ ]:
# Make predictions on the test set
y_pred=log_reg.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

# Calculate evalution metrics

accuracy=accuracy_score(y_test,y_pred)
precision=precision_score(y_test,y_pred, average='micro')
recall=recall_score(y_test,y_pred, average='micro')
f1=f1_score(y_test,y_pred, average='micro')
conf_martix = confusion_matrix(y_test,y_pred)

In [ ]:
# Print the results
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision : {precision:.2f}")
print(f"Recall : {recall:.2f}")
print(f"F1-score : {f1:.2f}")
print("Confusion Matrix:")
print(conf_martix)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(track_features, other_tracks):
    similarities = cosine_similarity(track_features.reshape(1, -1), other_tracks)
    return similarities.flatten()

In [ ]:
from scipy.spatial.distance import euclidean

# Calculate Euclidean distance
distance = euclidean(song1_features, song2_features)
print(distance)

In [ ]:
from scipy.stats import pearsonr

# Calculate Pearson correlation
correlation, _ = pearsonr(song1_features, song2_features)
print(correlation)

In [ ]:
from scipy.spatial.distance import cityblock

# Calculate Manhattan distance
distance = cityblock(song1_features, song2_features)
print(distance)

In [ ]:
!pip install dtw-python

In [ ]:
import numpy as np
from dtw import dtw

def dtw_distance_func(sequence1, sequence2):
    distance, _, _, _ = dtw(sequence1, sequence2, dist=lambda x, y: np.linalg.norm(x - y, ord=1))
    return distance

In [ ]:
print(dtw_distance_func)

In [ ]:
from sklearn.metrics import jaccard_score

def jaccard_similarity_func(set1, set2):
    return jaccard_score(set1, set2, average='macro')

In [ ]:
print (jaccard_similarity_func)

In [ ]:
def sequence_similarity(sequence1, sequence2, similarity_func):
    similarities = []
    for vector1, vector2 in zip(sequence1, sequence2):
        similarity = similarity_func(vector1, vector2)
        similarities.append(similarity)
    return np.mean(similarities)

In [ ]:
print(sequence_similarity)

In [ ]:
print(f"Input sequence shape: {input_sequence.shape}")

In [ ]:
print(f"Model input shape: {model.input_shape}")

In [ ]:
input_sequence = input_sequence.reshape((1, input_sequence.shape[1], input_sequence.shape[2]))
print(f"Reshaped input sequence shape: {input_sequence.shape}")

In [ ]:
sequence_length = 10
num_features = 10

predicted_features = model.predict(input_sequence)
print(f"Predicted features shape: {predicted_features.shape}")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def similarity_3d_averaging(array1, array2):
    # Average across the third dimension
    array1_avg = np.mean(array1, axis=1)
    array2_avg = np.mean(array2, axis=1)

    # Calculate cosine similarity
    return cosine_similarity(array1_avg, array2_avg)

# Example usage
array1 = np.random.rand(10, 5, 100)
array2 = np.random.rand(15, 5, 100)

similarity_scores = similarity_3d_averaging(array1, array2)
print(similarity_scores)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
y = y.values.reshape(-1,1)

In [ ]:
y.shape

In [ ]:
all_tracks_features = music_df[['Duration (ms)', 'Popularity', 'Acousticness', 'Danceability', 'Energy', 'Instrumentalness', 'Liveness', 'Loudness', 'Speechiness', 'Tempo']].values

In [ ]:
def recommend_track(model, input_sequence, all_tracks_features):
    # Predict features for the input sequence
    predicted_features = model.predict(input_sequence)

    # Reshape predicted_features to 2D if necessary
    if predicted_features.ndim == 3:
        predicted_features = predicted_features.reshape(predicted_features.shape[2], -1)

    # Reshape all_tracks_features to 2D
    all_tracks_features_reshaped = all_tracks_features.reshape(all_tracks_features.shape[0],-1)

    # Calculate cosine similarity between predicted features and all tracks
    similarity_scores = cosine_similarity(predicted_features, all_tracks_features_reshaped)

    # Find the index of the most similar track (considering all rows)
    most_similar_index = similarity_scores.argmax()
    row_index = most_similar_index // similarity_scores.shape[1]
    col_index = most_similar_index % similarity_scores.shape[1]
    similarity_score = similarity_scores[row_index, col_index]

    return most_similar_index, similarity_score

# Call the function with the correct number of arguments
most_similar_index, similarity_score = recommend_track(model, input_sequence, all_tracks_features)
print(f"Most similar track index: {most_similar_index} with similarity score: {similarity_score}")

In [ ]:
import pandas as pd

audio_features = []
for index, row in music_df.iterrows():
    track_features = {
        'Duration (ms)': row['Duration (ms)'],
        'Popularity': row['Popularity'],
    }
    audio_features.append(track_features)

df = pd.DataFrame(audio_features)
print(df)

In [ ]:
from IPython.display import Audio

In [ ]:
from IPython.display import IFrame

# Embed the Spotify playlist as an iframe
IFrame(src="https://open.spotify.com/embed/playlist/3r3PiLxBdeFCuxZYx3jGjx",
       width="800",
       height="880",
       frameborder="0",
       allowtransparency="true",
       autoplay = "full_songe",
       title="Arijit Singh playlist",
       allow="encrypted-media,decrypted-media")

In [ ]:
!pip install dash dash-bootstrap-components

In [ ]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc
import pandas as pd

# Create the Dash app
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Load or generate some example data
df = pd.DataFrame({
    'Track Name': ['Soulmate', 'Pal', 'Shayad', 'Enna Sona'],
    'Artists': ['Arijit Singh', 'Arijit Singh', 'Arijit Singh', 'Arijit Singh'],
    'Popularity': [50, 60, 70, 80]
})

# Define the layout of the app
app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Music Recommendation System"), className="mb-4")
    ]),
    dbc.Row([
        dbc.Col([
            html.Label('Select a Track'),
            dcc.Dropdown(
                id='track-dropdown',
                options=[{'label': row['Track Name'], 'value': index} for index, row in df.iterrows()],
                value=0
            )
        ], width=6),
        dbc.Col([
            html.Label('Popularity'),
            dcc.Slider(
                id='popularity-slider',
                min=0,
                max=100,
                step=1,
                value=50
            )
        ], width=6),
    ]),
    dbc.Row([
        dbc.Col(html.Div(id='track-info'))
    ]),
   dbc.Col([
            html.H3("Spotify Playlist"),
            html.Iframe(src="https://open.spotify.com/embed/playlist/3r3PiLxBdeFCuxZYx3jGjx",
                        width="800",
                        height="880",
                        # Remove the frameborder argument
                        style={'border': 'none'}, # Add this line to remove the frameborder using CSS
                        #allowtransparency="true", # This argument is deprecated
                        allow="encrypted-media")
        ])
], fluid=True) # This bracket was misaligned

# Define the callback to update the track information
@app.callback(
    Output('track-info', 'children'),
    [Input('track-dropdown', 'value'),
     Input('popularity-slider', 'value')]
)
def update_track_info(track_index, popularity_value):
    track = df.iloc[track_index]
    return html.Div([
        html.H4(f"Track Name: {track['Track Name']}"),
        html.P(f"Artists: {track['Artists']}"),
        html.P(f"Selected Popularity: {popularity_value}"),
    ])

# Run the app
if __name__ == '__main__':
    app.run_server(debug=True)